In [3]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
from langchain_core.documents import Document
from dotenv import load_dotenv
import os
load_dotenv()

True

In [4]:
# Creating Langchain Document for IPL Players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [5]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [6]:
# First we need to import the API ley from env and create a pinecone object
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

## Index details
#### Parameters:  
**name**: (Mandatory) Provide new index name  
**spec** - (Mandatory) Dict | ServerlessSpec | PodSpec | ByocSpec. All are classes, where we need to define additional parameters.  
`ServerlessSpec(cloud="Cloud_provider_name", region="Provide_cloud_region")`   
- `aws`  `us-east-1` (Virginia) --> Avaiable in free tier
- `aws`  `us-west-2` (Oregon)
- `aws`  `eu-west-1` (Ireland)
- `gcp`  `us-central1` (Iowa)
- `gcp`  `europe-west4` (Netherlands)
- `azure`  `eastus2` (Virginia)
  
**dimension** - Length of the vector. Larger dimension means larger storage space, meanshigher storage costs  
**metric** - which metric should be used to perform sematic operations. Options: *cosine* (default), *dotproduct*, *euclidean*  
**deletion_protection** - disabled by default  
**vector_type** - *dense* (default) or *sparse*

In [7]:
# Once the pc object is created, we can either create a new index (DB) or modify the existing one.
# Checking if the index is present
# Function used: pc.has_index()
# We are defining index name as 'langchain-demo' for example
index_name = "langchain-demo"
pc.has_index(index_name)

True

**If the above output is true, then use different name for creating a new index, or remove the old index. To use the same index, DO NOT run the create index code below**

In [80]:
# Creating a new index in Pinecone
# Function used: pc.create_index()

pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

{
    "name": "langchain-demo",
    "metric": "cosine",
    "host": "langchain-demo-wu411qn.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}

In [8]:
# Once the index is created, pull the index object from pinecone
# Here treat index as a DB
pc.has_index(index_name)
index = pc.Index(index_name)

In [9]:
# Once the index is created, we can create a vector store 
# Here we will use PineconeVectorStore class to create a vector store object.
# Pass the index name and embedding model to use

vector_store = PineconeVectorStore(
    index=index,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small")
)


In [10]:
## Adding documents in the DB
# Funtion used: add_documents()
# After exeuting this cell, it will show IDs created for each document embedding
vector_store.add_documents(documents=docs)

['7b9234b9-bd4f-444b-8af9-4d7f5c0b66c0',
 '25500836-22e2-4c34-9b44-38cc32d9ed4e',
 'e8a896e5-a05a-40e3-8cd4-8645545237a8',
 'fb8a39c7-4a4d-4781-8aeb-bf83e16a25d7',
 'bbe087fc-5f37-497d-a1b6-dc0db84c7ed6']

In [ ]:
# Performing similarity search
vector_store.similarity_search(
    query='Who is the bowler?',
    k=2
)

[Document(id='2994b743-5ac4-4f29-b3dc-6ca160385932', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='af305f28-6487-44f6-8c65-4cae7f0b68c7', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [89]:
# Search with similarity score
# Function used: similaritysearch_with_score()
# It provide similaarity scores along with the searched value
# Lower score means better result

vector_store.similarity_search_with_score(
    query='Who is batsman?',
    k=2
)

[(Document(id='2994b743-5ac4-4f29-b3dc-6ca160385932', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.437118143),
 (Document(id='af305f28-6487-44f6-8c65-4cae7f0b68c7', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.390664548)]

In [90]:
# Meta-data based search
# Add a parameter 'filter' and pass the dict (the key with the required value) in the metadata
vector_store.similarity_search_with_score(
    query="",
    filter={'team':'Chennai Super Kings'}
)

[(Document(id='af305f28-6487-44f6-8c65-4cae7f0b68c7', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.0896077156),
 (Document(id='7726d52f-9d87-49f9-816e-4857c35b8140', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.0866231173)]

In [18]:
## Modify/Update current document
# Function used: add_document()
# We are using the same method the update the document as well. If the id exists, it will overwrite, if it does not exits, it create a new one

# Creating a new document to update the current value 
updated_doc = Document(
    page_content='MS Dhoni, one of the best wicket-keep batsman ever in the cricket history. He has the fastest stumping records in the international cricket. Under his captionship Chennai Super Kings won 5 IPL trophies. He is the oldest creicketer still playing the IPL',
    metadata={'team': 'Chennai Super Kings'}
)

vector_store.add_documents(
    documents=[updated_doc],
    ids=["7726d52f-9d87-49f9-816e-4857c35b8140"]  # SAME ID → overwrite
)

['7726d52f-9d87-49f9-816e-4857c35b8140']

In [17]:
vector_store.similarity_search(
    query="MS Dhoni"
)

[Document(id='e8a896e5-a05a-40e3-8cd4-8645545237a8', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
 Document(id='7726d52f-9d87-49f9-816e-4857c35b8140', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, one of the best wicket-keep batsman ever in the cricket history. He has the fastest stumping records in the international cricket. Under his captionship Chennai Super Kings won 5 IPL trophies. He is the oldest creicketer still playing the IPL'),
 Document(id='bbe087fc-5f37-497d-a1b6-dc0db84c7ed6', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
 Document(id='d9d270e6-d0ec-41a1-8af1-c62e7337933f', metadata=

In [15]:
## Deleting data from VectorDB
# Function used: delete()
# Pass Document id to remove the record

# Removing data of Rebindra Jadeja
vector_store.delete(['af305f28-6487-44f6-8c65-4cae7f0b68c7'])